# Fetch Dataset & Column Descriptions

For every UID in `DatasetsWithSolidMetadata - Sheet1.csv`, fetch the dataset's metadata from
`https://{domain}/api/views/{uid}.json` and capture:

- the dataset description
- every (non-system) column's display name, field name, type, and description

Outputs:

1. `dataset_descriptions.json` — consolidated, keyed by UID (also acts as the resume cache)
2. `dataset_descriptions.jsonl` — one line per dataset, suitable as fine-tuning examples
3. `column_descriptions.jsonl` — one line per column, with parent-dataset context

Re-running the fetch cell is safe: anything already in `dataset_descriptions.json` is skipped
so only previously-failed datasets are retried. Set `FORCE_REFETCH = True` in **Settings** to
start over from scratch.

## Settings

In [ ]:
from pathlib import Path

EVAL_DIR = Path.cwd()  # this notebook lives in scripts/
CSV_PATH = EVAL_DIR / "DatasetsWithSolidMetadata - Sheet1.csv"
OUTPUT_JSON = EVAL_DIR / "dataset_descriptions.json"
DATASETS_JSONL = EVAL_DIR / "dataset_descriptions.jsonl"
COLUMNS_JSONL = EVAL_DIR / "column_descriptions.jsonl"

DEFAULT_DOMAIN = "data.wa.gov"
REQUEST_TIMEOUT = 30.0
SLEEP_BETWEEN = 0.1  # seconds — be polite to the portal

# Set True to ignore any existing dataset_descriptions.json and refetch everything.
FORCE_REFETCH = False

## Imports

In [ ]:
import csv
import json
import os
import sys
import time
from pathlib import Path

import httpx

# Add the repo root to sys.path and import backend.config for its side effect:
# it calls load_dotenv() on backend/.env, so SOCRATA_APP_TOKEN (and friends)
# are populated for os.getenv() below even when Jupyter was launched from a
# shell that hadn't sourced the env files.
sys.path.insert(0, str(Path.cwd().parent))
import backend.config  # noqa: F401

## Read the CSV

In [ ]:
with CSV_PATH.open(newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if (r.get("UID") or "").strip()]

print(f"{len(rows)} datasets listed in CSV.")

## Helpers

In [ ]:
def build_headers() -> dict[str, str]:
    # An app token raises Socrata rate limits, but isn't required for reads.
    headers = {"Accept": "application/json"}
    token = os.getenv("SOCRATA_APP_TOKEN", "").strip()
    if token:
        headers["X-App-Token"] = token
    return headers


def fetch_metadata(client: httpx.Client, domain: str, uid: str) -> dict:
    resp = client.get(f"https://{domain}/api/views/{uid}.json")
    resp.raise_for_status()
    return resp.json()


def extract(metadata: dict) -> dict:
    columns_out = []
    for col in metadata.get("columns") or []:
        field_name = col.get("fieldName") or ""
        # Skip Socrata system columns like :created_at, :updated_at.
        if field_name.startswith(":"):
            continue
        columns_out.append(
            {
                "fieldName": field_name,
                "name": col.get("name") or "",
                "description": col.get("description") or "",
                "dataTypeName": col.get("dataTypeName") or "",
            }
        )
    return {
        "name": metadata.get("name") or "",
        "description": metadata.get("description") or "",
        "columns": columns_out,
    }

## Fetch (with resume)

Reads any previously written `dataset_descriptions.json` and only fetches UIDs that aren't
already cached. Failures are collected and printed at the end so you can rerun this cell to
retry just the misses.

In [ ]:
results: dict[str, dict] = {}
if OUTPUT_JSON.exists() and not FORCE_REFETCH:
    try:
        prev = json.loads(OUTPUT_JSON.read_text())
        results = dict(prev.get("datasets") or {})
    except json.JSONDecodeError:
        print(f"Warning: {OUTPUT_JSON.name} is not valid JSON — starting fresh.")

todo = [r for r in rows if r["UID"].strip() not in results]
skipped = len(rows) - len(todo)
if skipped:
    print(f"Resuming: {skipped} already fetched, {len(todo)} to go.")
else:
    print(f"Fetching metadata for {len(todo)} datasets...")

errors: list[dict] = []
with httpx.Client(timeout=REQUEST_TIMEOUT, headers=build_headers()) as client:
    for i, row in enumerate(todo, start=1):
        uid = row["UID"].strip()
        domain = (row.get("Domain") or "").strip() or DEFAULT_DOMAIN
        try:
            metadata = fetch_metadata(client, domain, uid)
            results[uid] = {"domain": domain, **extract(metadata)}
            print(f"  [{i}/{len(todo)}] {uid} ok")
        except httpx.HTTPStatusError as e:
            msg = f"HTTP {e.response.status_code}"
            errors.append({"uid": uid, "domain": domain, "error": msg})
            print(f"  [{i}/{len(todo)}] {uid} FAILED: {msg}")
        except Exception as e:
            errors.append({"uid": uid, "domain": domain, "error": str(e)})
            print(f"  [{i}/{len(todo)}] {uid} FAILED: {e}")
        time.sleep(SLEEP_BETWEEN)

print(f"\n{len(results)} datasets cached, {len(errors)} failed this run.")

## Save consolidated JSON

This is the canonical output and the resume cache for future runs.

In [ ]:
output = {"datasets": results, "errors": errors}
OUTPUT_JSON.write_text(json.dumps(output, indent=2, ensure_ascii=False))
print(f"Wrote {len(results)} datasets to {OUTPUT_JSON.name}")

## Split into fine-tuning files

Two JSONL files — one per dataset, one per column. Each line is a self-contained JSON record
with enough context to be reshaped into chat / completion format by your fine-tuning pipeline
(e.g. wrap `description` as the `assistant` message and the rest as a `user` message).

Records with empty descriptions are skipped — they aren't useful training targets.

In [ ]:
# Dataset-level: one record per dataset with its description as the target.
# Column names/types travel with it as context so the model can ground the description.
dataset_examples = 0
with DATASETS_JSONL.open("w", encoding="utf-8") as f:
    for uid, ds in results.items():
        description = (ds.get("description") or "").strip()
        if not description:
            continue
        record = {
            "uid": uid,
            "domain": ds.get("domain", ""),
            "name": ds.get("name", ""),
            "columns": [
                {"name": c["name"], "dataTypeName": c["dataTypeName"]}
                for c in ds.get("columns", [])
            ],
            "description": description,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        dataset_examples += 1

print(f"Wrote {dataset_examples} dataset examples to {DATASETS_JSONL.name}")

In [ ]:
# Column-level: one record per column with parent-dataset context.
# datasetDescription is included so the model has the same context a human writer would.
column_examples = 0
with COLUMNS_JSONL.open("w", encoding="utf-8") as f:
    for uid, ds in results.items():
        for col in ds.get("columns", []):
            description = (col.get("description") or "").strip()
            if not description:
                continue
            record = {
                "uid": uid,
                "datasetName": ds.get("name", ""),
                "datasetDescription": ds.get("description", ""),
                "fieldName": col.get("fieldName", ""),
                "name": col.get("name", ""),
                "dataTypeName": col.get("dataTypeName", ""),
                "description": description,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            column_examples += 1

print(f"Wrote {column_examples} column examples to {COLUMNS_JSONL.name}")

## Summary

In [ ]:
total_cols = sum(len(ds.get("columns", [])) for ds in results.values())
described_ds = sum(
    1 for ds in results.values() if (ds.get("description") or "").strip()
)
described_cols = sum(
    1
    for ds in results.values()
    for c in ds.get("columns", [])
    if (c.get("description") or "").strip()
)

print(f"Datasets cached:           {len(results)}")
print(f"  with descriptions:       {described_ds}")
print(f"Columns cached:            {total_cols}")
print(f"  with descriptions:       {described_cols}")
print(f"Errors this run:           {len(errors)}")